# Refined Consensus Table

Build a fixture/market/selection-level consensus table from OddsJam odds with stable derived keys and moneyline-specific competitiveness features.

**Design goals**
- One row per unique market selection context within a fixture.
- Preserve broad market compatibility (not just moneyline).
- Precompute moneyline implied probability and competitiveness for match-length modeling.

In [1]:
import sys
sys.path.insert(0, "/Users/matt.holden/Projects/tennis-origination")

import pandas as pd

from injestion.core.bq import get_client
from injestion.oddsjam import OddsJamManager

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [2]:
bq_client = get_client()
oj_manager = OddsJamManager()

ODDS_TABLE_ID = oj_manager.get_table_id("oddsjam_odds")
FIXTURES_TABLE_ID = oj_manager.get_table_id("oddsjam_fixtures")

print(f"Odds table: {ODDS_TABLE_ID}")
print(f"Fixtures table: {FIXTURES_TABLE_ID}")

Odds table: prizepicksanalytics.originations_tennis.oj_odds
Fixtures table: prizepicksanalytics.originations_tennis.oj_fixtures


In [3]:
def get_refined_consensus_query(odds_table_id: str, fixtures_table_id: str) -> str:
    return f"""
WITH fixture_base AS (
  SELECT
    id AS fixture_id,
    start_date,
    league_name,
    status,
    result_json,
    SAFE_CAST(
      SAFE_CAST(JSON_VALUE(result_json, '$.scores.home.total') AS FLOAT64) AS INT64
    ) AS home_sets_won,
    SAFE_CAST(
      SAFE_CAST(JSON_VALUE(result_json, '$.scores.away.total') AS FLOAT64) AS INT64
    ) AS away_sets_won
  FROM `{fixtures_table_id}`
  WHERE league_name IN ('ATP', 'WTA')
),
fixture_with_match_length AS (
  SELECT
    fixture_id,
    start_date,
    league_name,
    status,
    home_sets_won,
    away_sets_won,
    CASE
      WHEN home_sets_won IS NULL OR away_sets_won IS NULL THEN NULL
      ELSE home_sets_won + away_sets_won
    END AS total_sets_from_result,
    CASE
      WHEN GREATEST(home_sets_won, away_sets_won) = 2 THEN 3
      WHEN GREATEST(home_sets_won, away_sets_won) = 3 THEN 5
      ELSE NULL
    END AS mode_best_of
  FROM fixture_base
),
odds_base AS (
  SELECT
    fixture_id,
    league_name,
    market,
    market_id,
    name,
    selection,
    normalized_selection,
    selection_line,
    player_id,
    team_id,
    sportsbook,
    is_main,
    closing_line_price,
    closing_line_points,
    CASE
      WHEN normalized_selection IS NULL THEN 'unknown_selection'
      WHEN TRIM(normalized_selection) = '' THEN 'combined_market'
      ELSE LOWER(TRIM(normalized_selection))
    END AS normalized_selection_key,
    CASE
      WHEN NULLIF(TRIM(selection_line), '') IS NOT NULL
        THEN LOWER(TRIM(selection_line))
      ELSE 'no_selection'
    END AS selection_line_key
  FROM `{odds_table_id}`
  WHERE league_name IN ('ATP', 'WTA')
    AND closing_line_price IS NOT NULL
    AND IFNULL(no_odds, FALSE) = FALSE
    AND market IS NOT NULL
),
consensus_by_key AS (
  SELECT
    fixture_id,
    league_name,
    market,
    market_id,
    normalized_selection_key,
    selection_line_key,
    closing_line_points,
    ANY_VALUE(name) AS name,
    ANY_VALUE(selection) AS selection,
    ANY_VALUE(normalized_selection) AS normalized_selection,
    ANY_VALUE(selection_line) AS selection_line,
    ANY_VALUE(player_id) AS player_id,
    ANY_VALUE(team_id) AS team_id,
    AVG(closing_line_price) AS avg_closing_line_price,
    AVG(closing_line_points) AS avg_closing_line_points
  FROM odds_base
  GROUP BY
    fixture_id,
    league_name,
    market,
    market_id,
    normalized_selection_key,
    selection_line_key,
    closing_line_points
)
SELECT
  TO_HEX(
    SHA256(
      CONCAT(
        IFNULL(c.fixture_id, ''), '||',
        IFNULL(c.market_id, c.market), '||',
        IFNULL(c.normalized_selection_key, ''), '||',
        IFNULL(c.selection_line_key, ''), '||',
        IFNULL(CAST(c.closing_line_points AS STRING), 'no_points')
      )
    )
  ) AS consensus_row_key,
  c.fixture_id,
  COALESCE(c.league_name, f.league_name) AS league_name,
  f.start_date,
  f.status,
  c.market,
  c.market_id,
  c.name,
  c.selection,
  c.normalized_selection,
  c.selection_line,
  c.normalized_selection_key,
  c.selection_line_key,
  c.player_id,
  c.team_id,
  c.closing_line_points,
  c.avg_closing_line_points,
  c.avg_closing_line_price,
  CASE
    WHEN c.avg_closing_line_price > 0 THEN 100.0 / (c.avg_closing_line_price + 100.0)
    WHEN c.avg_closing_line_price < 0 THEN (-c.avg_closing_line_price) / ((-c.avg_closing_line_price) + 100.0)
    ELSE NULL
  END AS implied_win_prob,
  CASE
    WHEN LOWER(c.market) = 'moneyline' THEN ABS(
      0.5 - (
        CASE
          WHEN c.avg_closing_line_price > 0 THEN 100.0 / (c.avg_closing_line_price + 100.0)
          WHEN c.avg_closing_line_price < 0 THEN (-c.avg_closing_line_price) / ((-c.avg_closing_line_price) + 100.0)
          ELSE NULL
        END
      )
    )
    ELSE NULL
  END AS moneyline_competitiveness_metric,
  f.mode_best_of,
  f.home_sets_won,
  f.away_sets_won,
  f.total_sets_from_result
FROM consensus_by_key c
LEFT JOIN fixture_with_match_length f
  ON f.fixture_id = c.fixture_id
ORDER BY f.start_date DESC, c.fixture_id
"""


consensus_query = get_refined_consensus_query(
    odds_table_id=ODDS_TABLE_ID,
    fixtures_table_id=FIXTURES_TABLE_ID,
)


In [4]:
# Build the refined consensus dataframe.
refined_consensus_df = bq_client.query(consensus_query).to_dataframe()

print(f"Rows: {len(refined_consensus_df):,}")
refined_consensus_df.head(20)

Rows: 582,979


,consensus_row_key,fixture_id,league_name,start_date,status,market,market_id,name,selection,normalized_selection,selection_line,normalized_selection_key,selection_line_key,player_id,team_id,closing_line_points,avg_closing_line_points,avg_closing_line_price,implied_win_prob,moneyline_competitiveness_metric,mode_best_of,home_sets_won,away_sets_won,total_sets_from_result
0,ad462c0f049a6994b22f166d5197e3943a5e456424716b...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,Player Sets Won,player_sets_won,Carolyn Ansari under,Carolyn Ansari,carolyn_ansari,under,carolyn_ansari,under,None,0C984D2DC1C53027,0.5,0.5,-263.000000,0.724518,NaN,<NA>,<NA>,<NA>,<NA>
1,6a85a5181437c842a0ef6f852891aef223820121f2634a...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,Moneyline,moneyline,Carolyn Ansari,Carolyn Ansari,carolyn_ansari,None,carolyn_ansari,no_selection,None,0C984D2DC1C53027,NaN,NaN,557.166667,0.152168,0.347832,<NA>,<NA>,<NA>,<NA>
2,630119b6c355fc8add9948f8fc8794c4c2cd461204e233...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,Player Games Won,player_games_won,Carolyn Ansari over,Carolyn Ansari,carolyn_ansari,over,carolyn_ansari,over,None,0C984D2DC1C53027,6.5,6.5,-111.000000,0.526066,NaN,<NA>,<NA>,<NA>,<NA>
3,cfceba0c43b20b42b3f520fec3cfd16b9c5fe539548bf4...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,Player Double Faults,player_double_faults,Carolyn Ansari over,Carolyn Ansari,carolyn_ansari,over,carolyn_ansari,over,None,0C984D2DC1C53027,4.5,4.5,110.000000,0.476190,NaN,<NA>,<NA>,<NA>,<NA>
4,ce12b2e273cf0e36a10956b96fc3500bb79d10e0e9a95d...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,1st Set Player Games Won,1st_set_player_games_won,Carolyn Ansari over,Carolyn Ansari,carolyn_ansari,over,carolyn_ansari,over,None,0C984D2DC1C53027,2.5,2.5,-161.000000,0.616858,NaN,<NA>,<NA>,<NA>,<NA>
5,4c18d96ef3913356e89304bcd6ba865e53409a3ae0b964...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,1st Set Player Games Won,1st_set_player_games_won,Darja Vidmanova over,Darja Vidmanova,darja_vidmanova,over,darja_vidmanova,over,None,8724BB122DDDC562,5.5,5.5,-667.000000,0.869622,NaN,<NA>,<NA>,<NA>,<NA>
6,6a2cca441fc732310c3f6feb0edbd5b7a38be74d93bc30...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,Player Sets Won,player_sets_won,Darja Vidmanova under,Darja Vidmanova,darja_vidmanova,under,darja_vidmanova,under,None,8724BB122DDDC562,0.5,0.5,1200.000000,0.076923,NaN,<NA>,<NA>,<NA>,<NA>
7,4b3e3c67e945c04a1c7255fd3c6fab3048676aee41f953...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,Total Aces,total_aces,over,,,over,combined_market,over,None,None,7.5,7.5,120.000000,0.454545,NaN,<NA>,<NA>,<NA>,<NA>
8,f670d43c390d85467801294b2cc719fa451a45203ccfa0...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,Player Aces + Double Faults,player_aces_+_double_faults,Carolyn Ansari over,Carolyn Ansari,carolyn_ansari,over,carolyn_ansari,over,None,0C984D2DC1C53027,6.5,6.5,112.000000,0.471698,NaN,<NA>,<NA>,<NA>,<NA>
9,8bd69c52cdc36cb8887d22d151d8e7b0fbe41c935ba452...,2026070685C7C714,WTA,2026-07-08 14:00:00+00:00,unplayed,Player Aces,player_aces,Carolyn Ansari over,Carolyn Ansari,carolyn_ansari,over,carolyn_ansari,over,None,0C984D2DC1C53027,1.5,1.5,-143.000000,0.588477,NaN,<NA>,<NA>,<NA>,<NA>


In [5]:
# Basic quality checks for uniqueness and expected key behavior.
unique_key_cols = [
    "fixture_id",
    "market",
    "market_id",
    "normalized_selection_key",
    "selection_line_key",
    "closing_line_points",
]

dupe_count = int(refined_consensus_df.duplicated(subset=unique_key_cols).sum())
print(f"Duplicate rows under expected unique key: {dupe_count}")

if dupe_count > 0:
    display(
        refined_consensus_df[
            refined_consensus_df.duplicated(subset=unique_key_cols, keep=False)
        ].sort_values(unique_key_cols)
    )

norm_raw = refined_consensus_df["normalized_selection"].astype("string")
norm_trim = norm_raw.fillna("").str.strip()

null_norm_count = int(norm_raw.isna().sum())
empty_norm_count = int((norm_trim == "").sum())

print(f"\nRaw normalized_selection NULL rows: {null_norm_count}")
print(f"Raw normalized_selection empty-string rows: {empty_norm_count}")

if null_norm_count > 0:
    print("Rows with NULL normalized_selection by market")
    display(
        refined_consensus_df[norm_raw.isna()]
        .value_counts("market")
        .rename("rows")
        .to_frame()
    )

invalid_null_mapping = int(
    ((norm_raw.isna()) & (refined_consensus_df["normalized_selection_key"] != "unknown_selection")).sum()
)
invalid_empty_mapping = int(
    ((norm_trim == "") & (refined_consensus_df["normalized_selection_key"] != "combined_market")).sum()
)
print(f"NULL -> unknown_selection mapping violations: {invalid_null_mapping}")
print(f"Empty string -> combined_market mapping violations: {invalid_empty_mapping}")

print("\nRows by market")
display(refined_consensus_df.value_counts("market").head(25).rename("rows").to_frame())

print("\nRows with normalized_selection_key='combined_market'")
print(int((refined_consensus_df["normalized_selection_key"] == "combined_market").sum()))

print("\nRows with selection_line_key='no_selection'")
print(int((refined_consensus_df["selection_line_key"] == "no_selection").sum()))

Duplicate rows under expected unique key: 0

Raw normalized_selection NULL rows: 0
Raw normalized_selection empty-string rows: 253033
NULL -> unknown_selection mapping violations: 0
Empty string -> combined_market mapping violations: 0

Rows by market


,rows
market,
Player Games Won,105649
Total Games,82667
1st Set Total Games,62576
1st Set Player Games Won,45700
Player Sets Won,42950
Total Tie Breaks,38593
Player Aces,34403
Player Double Faults,33107
Total Sets,32285



Rows with normalized_selection_key='combined_market'
253033

Rows with selection_line_key='no_selection'
38920


In [6]:
# Moneyline spot-check: favorite selection per fixture and competitiveness metric.
moneyline_df = refined_consensus_df[
    refined_consensus_df["market"].fillna("").str.lower() == "moneyline"
].copy()

moneyline_favorites = (
    moneyline_df.sort_values(["fixture_id", "implied_win_prob"], ascending=[True, False])
    .drop_duplicates(subset=["fixture_id"])
)

print(f"Moneyline rows: {len(moneyline_df):,}")
print(f"Favorite rows by fixture: {len(moneyline_favorites):,}")

display(
    moneyline_favorites[
        [
            "fixture_id",
            "league_name",
            "selection",
            "avg_closing_line_price",
            "implied_win_prob",
            "moneyline_competitiveness_metric",
            "mode_best_of",
            "total_sets_from_result",
        ]
    ].head(20)
)

Moneyline rows: 31,957
Favorite rows by fixture: 15,959


,fixture_id,league_name,selection,avg_closing_line_price,implied_win_prob,moneyline_competitiveness_metric,mode_best_of,total_sets_from_result
581094,0038CFAB1598,ATP,Arthur Fils,-391.875000,0.796696,0.296696,5,5
577496,0372116C47CB,ATP,Frances Tiafoe,-403.000000,0.801193,0.301193,3,2
576744,03F6F3F335E9,WTA,Diana Shnaider,-567.272727,0.850136,0.350136,3,3
581974,061F7DDF065B,WTA,Petra Martic,-146.777778,0.594777,0.094777,3,2
579342,063EC2E79B42,WTA,Sara Saito,-278.600000,0.735869,0.235869,3,3
580602,079BAFA9454C,WTA,Heather Watson,-263.300000,0.724745,0.224745,3,2
580885,080FBFF06636,ATP,Joao Fonseca,-124.666667,0.554896,0.054896,5,5
577636,08A6A72C24C7,ATP,Alexei Popyrin,-176.000000,0.637681,0.137681,3,2
579540,09A76016F167,WTA,Veronika Erjavec,-134.500000,0.573561,0.073561,3,2
579159,09FCB0751A54,WTA,Marie Bouzkova,-485.909091,0.829325,0.329325,3,3


In [7]:
refined_consensus_df.dtypes

consensus_row_key                                object
fixture_id                                       object
league_name                                      object
start_date                          datetime64[us, UTC]
status                                           object
market                                           object
market_id                                        object
name                                             object
selection                                        object
normalized_selection                             object
selection_line                                   object
normalized_selection_key                         object
selection_line_key                               object
player_id                                        object
team_id                                          object
closing_line_points                             float64
avg_closing_line_points                         float64
avg_closing_line_price                          

## Load to BigQuery (after review)

Keep this cell commented out until we're ready to publish the first table version.

In [8]:
from injestion.core.bq import get_client
from refined_tables import replace_table, get_table_id
from refined_tables.schema import consensus as schema_consensus

client = get_client()
replace_table(
    client,
    get_table_id("consensus"),
    refined_consensus_df,
    schema=schema_consensus.get_schema(),
)

/Users/matt.holden/Projects/tennis-origination/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prizepicksanalytics, location=US, id=0658d498-5a70-4988-8b47-3d2d6add2964>